# IMPLEMENTATION

## 1\. ⚙️ Data Collection and Sourcing

This initial stage focuses on gathering all the necessary historical data.

  * **Action:** Collect time series data for the two selected currency exchange rates (your dependent variable) and all key features (interest rate cuts, inflation, job data, unemployment, and proxy indicators for news/tariffs/global conflicts) for the relevant countries and global context.
  * **Explanation:** Macroeconomic data is typically sourced from central banks, statistical agencies, and financial data providers. Geopolitical and sentiment data often requires **Natural Language Processing (NLP)** techniques on news articles or the use of established indexes like the Geopolitical Risk (GPR) Index or Economic Policy Uncertainty (EPU) Index.


## 📋 List of Features to consider

Here is a consolidated, copyable list of the features organized by category, suitable for direct use in documentation or coding.

---

### 1. 📈 Fundamental (Macroeconomic) Indicators

* Interest Rate Differential
* Inflation Rate (CPI, Core CPI)
* Gross Domestic Product (GDP) Growth Rate
* Unemployment Rate
* Non-Farm Payrolls (NFP) or Primary Job Creation Figure
* Purchasing Managers' Index (PMI) (Manufacturing and Services)
* Trade Balance / Current Account Balance

---

### 2. 🌍 Global & Geopolitical/Sentiment Features

* Geopolitical Risk (GPR) Index
* Economic Policy Uncertainty (EPU) Index
* Global Market Volatility (VIX Index)
* Commodity Prices (e.g., Oil, Gold, Copper)
* Central Bank Policy Tone (Hawkish/Dovish Score via NLP)
* Retail/Institutional Sentiment Index for the Currency Pair

---

### 3. 📉 Market-Based (Technical) Features

* Lagged Exchange Rate Values (e.g., previous day's close)
* Simple Moving Average (SMA)
* Exponential Moving Average (EMA)
* Historical Volatility (Standard Deviation of Returns)
* Relative Strength Index (RSI)
* Momentum/Rate of Change (ROC)
* Average True Range (ATR)

In [6]:
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import pandas_ta_classic as ta
from datetime import datetime

# --- 1. CONFIGURATION ---
START_DATE = datetime(2005, 1, 1)
END_DATE = datetime.today()
# Example Currencies: USD and EUR
CURRENCY_PAIR_TICKER = 'EURUSD=X' # Yahoo Finance ticker for EUR/USD
COUNTRY_A = 'USA'
COUNTRY_B = 'Euro Zone'

# FRED Tickers for US (Example Country A)
# Use a similar set of codes for Country B if available on FRED or another source
FRED_CODES = {
    # Fundamental Indicators (US examples)
    'FEDFUNDS': 'US_Interest_Rate',     # Interest Rate (Federal Funds Effective Rate)
    'CPIAUCSL': 'US_Inflation',         # Inflation (CPI)
    'UNRATE': 'US_Unemployment',        # Unemployment Rate
    'PAYEMS': 'US_NonFarm_Payrolls',    # Job Data
    'GDPC1': 'US_GDP',                  # GDP (Real Gross Domestic Product)
    'BOPGSTB': 'US_Trade_Balance',      # Trade Balance

    # Global/Sentiment Indicators
    'VIXCLS': 'Global_VIX',             # Global Volatility (CBOE VIX)
    'DCOILWTICO': 'WTI_Crude_Oil_Price' # Commodity Price (Global Indicator)
    
    # Note: GPR and EPU often require direct downloads from academic websites
    # or specific APIs, as they are not reliably in FRED or pandas-datareader.
}


# --- 2. DATA DOWNLOAD FUNCTIONS ---

def get_exchange_rate(ticker, start, end):
    """Downloads the exchange rate using yfinance (since pdr is limited for Forex)."""
    print(f"Downloading {ticker} data...")
    try:
         s = yf.download(
            ticker,
            start=start,
            end=end,
            interval='1d'
        )['Close']

        # If MultiIndex (Ticker + Date), remove the Ticker level
         if isinstance(s.index, pd.MultiIndex) and s.index.nlevels > 1:
            s = s.droplevel(0)

        # Rename safely
         s.name = "Close_Price"

         return pd.DataFrame(s)
        # data = yf.download(
        #     ticker,
        #     start=start,
        #     end=end,
        #     interval='1d'   # FIXED
        # )['Close'].rename('Close_Price')
        # return pd.DataFrame(data)
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")
        return pd.DataFrame()

def get_fred_data(tickers, start, end):
    """Downloads macroeconomic and global indicators from FRED."""
    print("Downloading FRED data...")
    try:
        # Fetch all series at once
        fred_data = web.DataReader(list(tickers.keys()), 'fred', start, end)
        # Rename columns for clarity
        fred_data.columns = list(tickers.values())
        return fred_data
    except Exception as e:
        print(f"Error downloading FRED data: {e}")
        return pd.DataFrame()


# --- 3. TECHNICAL INDICATOR CALCULATION ---

def add_technical_indicators(df):
    """Calculates Lagged Price, Moving Averages, Volatility, and RSI."""
    # Ensure the input DataFrame has a 'Close_Price' column
    if 'Close_Price' not in df.columns:
        print("Error: DataFrame must contain a 'Close_Price' column.")
        return df

    print("Calculating Technical Indicators...")
    
    # 3.1 Lagged Exchange Rate Values
    df['Lag_1D'] = df['Close_Price'].shift(1)
    
    # 3.2 Moving Averages (SMA)
    df['SMA_20'] = ta.sma(df['Close_Price'], length=20)
    df['SMA_50'] = ta.sma(df['Close_Price'], length=50)
    
    # 3.3 Relative Strength Index (RSI)
    df['RSI_14'] = ta.rsi(df['Close_Price'], length=14)
    
    # 3.4 Historical Volatility (using ATR as a proxy for market volatility)
    # ATR requires 'High' and 'Low' which are not in the 'Close_Price' data, so we'll use price std dev for simplicity
    df['Historical_Volatility_20D'] = df['Close_Price'].pct_change().rolling(window=20).std() * (252**0.5) # Annualized

    return df


# --- 4. MAIN EXECUTION ---

# Get Forex Data (Daily Frequency)
forex_df = get_exchange_rate(CURRENCY_PAIR_TICKER, START_DATE, END_DATE)
print(forex_df)
# Calculate Technical Indicators
forex_df = add_technical_indicators(forex_df)
print(forex_df)
# Get FRED Data (Lower Frequency - will need resampling/forward-filling later)
fred_df = get_fred_data(FRED_CODES, START_DATE, END_DATE)

# Combine the dataframes
# We use an outer join to keep all data, then resample/forward fill the lower frequency data
combined_df = forex_df.join(fred_df, how='outer')

# --- 5. DATA PREPROCESSING (SKELETON) ---
print("\n--- Preprocessing Steps (Skeleton) ---")
# Macroeconomic data (FRED) is lower frequency (Monthly/Quarterly). 
# We need to Forward Fill (ffill) the values so they can be used for daily prediction.
combined_df = combined_df.ffill()

# Drop rows with any remaining NaNs (e.g., initial period for technical indicators)
combined_df = combined_df.dropna()

print(f"Final combined DataFrame shape: {combined_df.shape}")
print(combined_df.tail())

C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\3601746391.py:41: FutureWarning: YF.download() has changed argument auto_adjust default to True
  s = yf.download(
[*********************100%***********************]  1 of 1 completed


Ticker      EURUSD=X
Date                
2005-01-03  1.347001
2005-01-04  1.328198
2005-01-05  1.328004
2005-01-06  1.318305
2005-01-07  1.306097
...              ...
2025-11-21  1.153456
2025-11-24  1.150761
2025-11-25  1.152007
2025-11-26  1.156484
2025-11-27  1.160214

[5423 rows x 1 columns]
Error: DataFrame must contain a 'Close_Price' column.
Ticker      EURUSD=X
Date                
2005-01-03  1.347001
2005-01-04  1.328198
2005-01-05  1.328004
2005-01-06  1.318305
2005-01-07  1.306097
...              ...
2025-11-21  1.153456
2025-11-24  1.150761
2025-11-25  1.152007
2025-11-26  1.156484
2025-11-27  1.160214

[5423 rows x 1 columns]

--- Preprocessing Steps (Skeleton) ---
Final combined DataFrame shape: (5524, 9)
            EURUSD=X  US_Interest_Rate  US_Inflation  US_Unemployment  \
2025-11-21  1.153456              4.09       324.368              4.4   
2025-11-24  1.150761              4.09       324.368              4.4   
2025-11-25  1.152007              4.09       324.

In [7]:
combined_df

,EURUSD=X,US_Interest_Rate,US_Inflation,US_Unemployment,US_NonFarm_Payrolls,US_GDP,US_Trade_Balance,Global_VIX,WTI_Crude_Oil_Price
2005-01-03,1.347001,2.28,191.600,5.3,132781.0,15844.727,-56189.0,14.08,42.16
2005-01-04,1.328198,2.28,191.600,5.3,132781.0,15844.727,-56189.0,13.98,43.96
2005-01-05,1.328004,2.28,191.600,5.3,132781.0,15844.727,-56189.0,14.09,43.41
2005-01-06,1.318305,2.28,191.600,5.3,132781.0,15844.727,-56189.0,13.58,45.51
2005-01-07,1.306097,2.28,191.600,5.3,132781.0,15844.727,-56189.0,13.49,45.32
...,...,...,...,...,...,...,...,...,...
2025-11-21,1.153456,4.09,324.368,4.4,159626.0,23770.976,-59550.0,23.43,58.86
2025-11-24,1.150761,4.09,324.368,4.4,159626.0,23770.976,-59550.0,20.52,59.11
2025-11-25,1.152007,4.09,324.368,4.4,159626.0,23770.976,-59550.0,18.56,59.11
2025-11-26,1.156484,4.09,324.368,4.4,159626.0,23770.976,-59550.0,18.56,59.11


In [33]:
import pandas as pd

# Load daily news sentiment
news_sentiment = pd.read_csv("sentiment_news.csv")
news_sentiment.columns = news_sentiment.columns.str.strip().str.lower()
news_sentiment['date'] = pd.to_datetime(news_sentiment['date'])
news_sentiment.set_index('date', inplace=True)

# Ensure daily macro data index is datetime
combined_df.index = pd.to_datetime(combined_df.index)

# Step 1: Resample both to weekly frequency (mean for numerical features)
news_sentiment_weekly = news_sentiment.resample('W').mean()
macro_weekly = combined_df.resample('W').mean()

# Step 2: Combine weekly data
df_combined = pd.concat([macro_weekly, news_sentiment_weekly], axis=1)

# Optional: check number of weeks combined
print(f"Number of weeks combined: {len(df_combined_weekly)}")
print(df_combined.head())


Number of weeks combined: 1091
            EURUSD=X  US_Interest_Rate  US_Inflation  US_Unemployment  \
2005-01-09  1.325521             2.280        191.60             5.30   
2005-01-16  1.315858             2.280        191.60             5.30   
2005-01-23  1.300681             2.280        191.60             5.30   
2005-01-30  1.303481             2.280        191.60             5.30   
2005-02-06  1.299058             2.456        192.24             5.38   

            US_NonFarm_Payrolls     US_GDP  US_Trade_Balance  Global_VIX  \
2005-01-09             132781.0  15844.727          -56189.0      13.844   
2005-01-16             132781.0  15844.727          -56189.0      12.850   
2005-01-23             132781.0  15844.727          -56189.0      13.254   
2005-01-30             132781.0  15844.727          -56189.0      13.726   
2005-02-06             132982.6  15844.727          -57713.0      11.902   

            WTI_Crude_Oil_Price  news_sentiment_directional  
2005-01-09 

np.int64(562)

## 2\. 🧹 Data Preprocessing and Feature Engineering

Raw data must be cleaned, aligned, and transformed to be suitable for time series modeling.

  * **Action:**
      * **Data Cleaning:** Handle missing values (e.g., using interpolation or forward/backward fill), address outliers, and ensure data types are correct.
      * **Synchronization/Resampling:** Since macroeconomic indicators are often released at different frequencies (daily, monthly, quarterly), the data must be resampled to a consistent time interval (e.g., weekly or monthly) for unified modeling.
      * **Stationarity Check:** For time series modeling, currency movements and macroeconomic data often need to be made **stationary** (mean, variance, and covariance do not change over time) using techniques like **differencing** or logarithmic transformations.
      * **Feature Engineering:** Create new, predictive features, such as **lagged variables** (past values of the exchange rate or indicators) or **differentials** (e.g., the difference in interest rates between the two countries).



In [40]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

# --- Configuration ---
TARGET_FREQ = 'M'  # Monthly frequency for resampling (e.g., 'W' for weekly, 'M' for monthly)
FX_COLUMN = 'EURUSD=X'
SENTIMENT_COLUMN = 'News_Sentiment_Directional'

# -------------------------------------------------------------------
# --- 1. Data Cleaning: Handles NaNs and Outliers (Numeric only) ---
# -------------------------------------------------------------------
def clean_data(df):
    """Handles NaNs (ffill/bfill) and Outliers (IQR capping) in numeric columns."""
    print("1. Data Cleaning: Handling NaNs and Outliers...")
    df_cleaned = df.copy()
    
    # 1. Handle NaNs across all columns (numeric and categorical)
    df_cleaned = df_cleaned.ffill().bfill() 
        
    # 2. Handle Outliers (ONLY for numeric columns)
    # Identify numeric columns, excluding the categorical sentiment column
    numeric_cols_for_outlier = [
        col for col in df_cleaned.select_dtypes(include=np.number).columns 
        if col != SENTIMENT_COLUMN
    ]

    for col in numeric_cols_for_outlier:
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        # Cap outliers to the defined boundary
        df_cleaned[col] = np.where(df_cleaned[col] < lower, lower, df_cleaned[col])
        df_cleaned[col] = np.where(df_cleaned[col] > upper, upper, df_cleaned[col])
    
    df_cleaned.dropna(inplace=True)
    print(f"    Data shape after cleaning: {df_cleaned.shape}")
    return df_cleaned

# -------------------------------------------------------------------
# --- 2. Resampling: Synchronizes frequency. Mode for Sentiment. ---
# -------------------------------------------------------------------
def resample_data(df, freq):
    """Resamples data to the target frequency, using appropriate aggregation methods."""
    print(f"2. Synchronization: Resampling to {freq} frequency...")
    df_resampled = pd.DataFrame(index=df.resample(freq).last().index)
    
    # Identify numeric columns for 'last' aggregation
    numeric_cols = [col for col in df.columns if col != SENTIMENT_COLUMN]
    for col in numeric_cols:
        # FX and Macro columns: take last known value in the period
        df_resampled[col] = df[col].resample(freq).last()
        
    # News Sentiment Directional (Categorical): take the mode (most frequent)
    if SENTIMENT_COLUMN in df.columns:
        # Custom function to get the mode; safely returns the first mode or NaN
        sentiment_mode = df[SENTIMENT_COLUMN].resample(freq).agg(
            lambda x: x.mode()[0] if not x.mode().empty else np.nan
        )
        df_resampled[SENTIMENT_COLUMN] = sentiment_mode

    df_resampled.ffill(inplace=True)
    print(f"    Data shape after resampling: {df_resampled.shape}")
    return df_resampled.dropna()

# -------------------------------------------------------------------
# --- 3. Stationarity Check & Differencing (Numeric only) ---
# -------------------------------------------------------------------
def check_and_difference(series, feature_name, alpha=0.05):
    """Performs ADF test and differences the series if non-stationary."""
    
    # Skip differencing for non-numeric columns like sentiment
    if series.dtype == 'object' or pd.api.types.is_categorical_dtype(series) or not pd.api.types.is_numeric_dtype(series):
        print(f"    * {feature_name}: Categorical/Skipped Differencing")
        return series.rename(feature_name)
    
    adf_result = adfuller(series.dropna(), autolag='AIC')
    p_value = adf_result[1]
    
    if p_value < alpha:
        print(f"    * {feature_name}: Stationary (p={p_value:.4f})")
        return series.rename(feature_name)
    else:
        # Attempt first-order differencing
        series_diff = series.diff().dropna()
        adf_diff = adfuller(series_diff, autolag='AIC')
        if adf_diff[1] < alpha:
            print(f"    * {feature_name}: Differenced (p={adf_diff[1]:.4f})")
            return series_diff.rename(f'{feature_name}_D1')
        else:
            # If still non-stationary, return the original 
            print(f"    * {feature_name}: Still non-stationary (p={p_value:.4f}). Returning original.")
            return series.rename(feature_name)

def make_stationary(df, cols_to_test):
    """Applies stationarity check and differencing to specified columns."""
    print("\n3. Stationarity Check & Differencing...")
    df_stationary = pd.DataFrame(index=df.index)
    
    for col in cols_to_test:
        if col in df.columns:
            processed_series = check_and_difference(df[col], col)
            df_stationary[processed_series.name] = processed_series
    
    # 1. Identify original numeric columns that need to be dropped/replaced
    numeric_cols_to_drop = [
        col for col in cols_to_test 
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]
    
    df_out = df.copy()
    
    # 2. Drop the original non-stationary columns (e.g., EURUSD=X)
    df_out = df_out.drop(columns=numeric_cols_to_drop, errors='ignore')
    
    # 3. Join the processed columns (e.g., EURUSD=X_D1) and the unchanged sentiment column
    # We use df_stationary to bring in the new columns (differenced or unchanged)
    df_out = df_out.join(df_stationary)
    
    return df_out.dropna()

# -------------------------------------------------------------------
# --- 4. Feature Engineering: Creating Lagged Variables (All columns) ---
# -------------------------------------------------------------------
def feature_engineer(df):
    """Creates Lag 1 and Lag 4 features for all columns."""
    print("\n4. Feature Engineering: Creating Lagged Variables...")
    df_fe = df.copy()
    
    all_cols = df_fe.columns.tolist()
    
    for col in all_cols:
        # Lag 1: Shift by 1 period (month/week)
        df_fe[f'{col}_Lag1'] = df_fe[col].shift(1)
        # Lag 4: Shift by 4 periods (quarterly/monthly equivalent)
        df_fe[f'{col}_Lag4'] = df_fe[col].shift(4)
    
    return df_fe.dropna()

# ===================================================================
# --- MAIN EXECUTION
# ===================================================================

# --- A. Create Dummy Data for Testing ---
# NOTE: This creates a DataFrame to ensure the code runs without error.
print(f"Original Data Shape: {df_combined.shape}")
print(f"Target Resampling Frequency: {TARGET_FREQ}\n")

# 1. Cleaning
df_clean = clean_data(df_combined)

# 2. Resampling
df_resampled = resample_data(df_clean, TARGET_FREQ)

# 3. Feature Engineering (lags) - Applied to all current columns
df_features = feature_engineer(df_resampled)

# 4. Stationarity check & Differencing (Applied only to non-lagged original columns)
cols_to_test_final = [
    col for col in df_features.columns 
    if not col.endswith('_Lag1') and not col.endswith('_Lag4')
]

df_final = make_stationary(df_features, cols_to_test_final)


# --- FINAL OUTPUT ---
print("\n" + "="*50)
print("--- FINAL CLEANED, RESAMPLED, AND ENGINEERED DATA ---")
print("="*50)
print(f"Final Data Shape: {df_final.shape}")
print("\nFirst 5 rows:")
print(df_final.head())
print("\nFinal Columns:")
print(df_final.columns.tolist())

Original Data Shape: (1091, 10)
Target Resampling Frequency: M

1. Data Cleaning: Handling NaNs and Outliers...
    Data shape after cleaning: (1091, 10)
2. Synchronization: Resampling to M frequency...
    Data shape after resampling: (251, 10)

4. Feature Engineering: Creating Lagged Variables...

3. Stationarity Check & Differencing...
    * EURUSD=X: Differenced (p=0.0000)
    * US_Interest_Rate: Differenced (p=0.0043)
    * US_Inflation: Differenced (p=0.0000)
    * US_Unemployment: Differenced (p=0.0000)
    * US_NonFarm_Payrolls: Differenced (p=0.0000)
    * US_GDP: Differenced (p=0.0000)
    * US_Trade_Balance: Differenced (p=0.0000)
    * Global_VIX: Stationary (p=0.0001)
    * WTI_Crude_Oil_Price: Stationary (p=0.0059)
    * news_sentiment_directional: Differenced (p=0.0000)

--- FINAL CLEANED, RESAMPLED, AND ENGINEERED DATA ---
Final Data Shape: (246, 30)

First 5 rows:
            EURUSD=X_Lag1  EURUSD=X_Lag4  US_Interest_Rate_Lag1  \
2005-06-30       1.256679       1.31914

C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\2823600336.py:48: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_resampled = pd.DataFrame(index=df.resample(freq).last().index)
C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\2823600336.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_resampled[col] = df[col].resample(freq).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\2823600336.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_resampled[col] = df[col].resample(freq).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\2823600336.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_resampled[col] = df[col].resample(freq).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_29952\2823600336.py:54: FutureWarning: 'M' is deprecated and wil

In [41]:
df_final.columns

Index(['EURUSD=X_Lag1', 'EURUSD=X_Lag4', 'US_Interest_Rate_Lag1',
       'US_Interest_Rate_Lag4', 'US_Inflation_Lag1', 'US_Inflation_Lag4',
       'US_Unemployment_Lag1', 'US_Unemployment_Lag4',
       'US_NonFarm_Payrolls_Lag1', 'US_NonFarm_Payrolls_Lag4', 'US_GDP_Lag1',
       'US_GDP_Lag4', 'US_Trade_Balance_Lag1', 'US_Trade_Balance_Lag4',
       'Global_VIX_Lag1', 'Global_VIX_Lag4', 'WTI_Crude_Oil_Price_Lag1',
       'WTI_Crude_Oil_Price_Lag4', 'news_sentiment_directional_Lag1',
       'news_sentiment_directional_Lag4', 'EURUSD=X_D1', 'US_Interest_Rate_D1',
       'US_Inflation_D1', 'US_Unemployment_D1', 'US_NonFarm_Payrolls_D1',
       'US_GDP_D1', 'US_Trade_Balance_D1', 'Global_VIX', 'WTI_Crude_Oil_Price',
       'news_sentiment_directional_D1'],
      dtype='object')

## 3\. 🧠 Predictive Model Development and Training

This stage involves selecting, building, and training the forecasting model.

  * **Action:** Choose a suitable time series forecasting model or machine learning algorithm. Given the complexity of the factors, techniques like **ARIMA/GARCH** (traditional econometric models), **Vector Autoregressive (VAR)** models, or modern **Machine Learning** models like **Random Forests**, **XGBoost**, or **Recurrent Neural Networks (RNNs/LSTMs)** are strong candidates. Split the data into training, validation, and test sets.
  * **Explanation:** The model will learn the relationship between the macroeconomic/geopolitical factors and the historical currency movements from the training data. The model is then tuned on the validation set to find the optimal **hyperparameters**.


In [42]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# --- Configuration ---
TARGET_COLUMN = 'EURUSD=X_D1'
FORECAST_HORIZON = 1
TEST_SIZE_FRACTION = 0.2

# --- 1. Data Preparation and Chronological Train/Test Split ---
def prepare_data_and_split(df, target_col, test_size_fraction):
    print("1. Splitting Data...")
    
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    valid_idx = y.dropna().index
    X = X.loc[valid_idx]
    y = y.loc[valid_idx]
    
    n_test = int(len(X) * test_size_fraction)
    
    X_train = X.iloc[:-n_test]
    y_train = y.iloc[:-n_test]
    
    X_test = X.iloc[-n_test:]
    y_test = y.iloc[-n_test:]
    
    print(f"   Train size: {len(X_train)}, Test size: {len(X_test)}")
    return X_train, X_test, y_train, y_test

# --- 2. Random Forest Model ---
def train_rf_model(X_train, y_train):
    print("\n2. Training Random Forest...")
    rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    return rf

# --- 3. Evaluate Model ---
def evaluate_model(model, X_test, y_test, target_name):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"\nEvaluation ({model.__class__.__name__}) on {target_name}:")
    print(f"   RMSE: {rmse:.6f}, R²: {r2:.4f}")
    
    return y_pred, rmse

# --- 4. VAR Model ---
def train_var_model(df, target_col):
    print("\n4. Training VAR Model...")
    # Use only stationary or differenced columns
    var_cols = [col for col in df.columns if '_D1' in col or 'Differential' in col]
    var_data = df[var_cols].dropna()
    
    model = VAR(var_data)
    lag_order = model.select_order(maxlags=10).selected_orders['aic']
    print(f"   Optimal VAR lag order: {lag_order}")
    
    var_results = model.fit(lag_order)
    return var_results, var_cols, lag_order

# --- MAIN EXECUTION ---
# Split data
X_train, X_test, y_train, y_test = prepare_data_and_split(df_final, TARGET_COLUMN, TEST_SIZE_FRACTION)

# Random Forest
rf_model = train_rf_model(X_train, y_train)
rf_pred, rf_rmse = evaluate_model(rf_model, X_test, y_test, TARGET_COLUMN)
y_test_rf = y_test
# VAR Model
var_results, var_cols, lag_order = train_var_model(df_final, TARGET_COLUMN)

# VAR Forecast (next step)
forecast = var_results.forecast(y=df_final[var_cols].values[-lag_order:], steps=FORECAST_HORIZON)
forecast_df = pd.DataFrame(forecast, columns=var_cols)
print("\nVAR Forecast (next step) for EURUSD=X_D1:")
print(forecast_df[[TARGET_COLUMN]])


1. Splitting Data...
   Train size: 197, Test size: 49

2. Training Random Forest...

Evaluation (RandomForestRegressor) on EURUSD=X_D1:
   RMSE: 0.021482, R²: -0.0017

4. Training VAR Model...
   Optimal VAR lag order: 6

VAR Forecast (next step) for EURUSD=X_D1:
   EURUSD=X_D1
0     0.004149


In [43]:
# --- Install required libraries if not installed ---
# pip install pytorch-lightning pytorch-forecasting torch torchvision torchaudio

import pandas as pd
import numpy as np
import torch
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch import Trainer, seed_everything

# -----------------------------
# 0. Reproducibility
# -----------------------------
seed_everything(42)

# -----------------------------
# 1. Configuration
# -----------------------------
TARGET = 'EURUSD=X_D1'
TIME_IDX = 'time_idx'
MAX_ENCODER_LENGTH = 12  # lookback history
MAX_PREDICTION_LENGTH = 1  # forecast horizon
BATCH_SIZE = 32
N_EPOCHS = 30

# -----------------------------
# 2. Prepare Data
# -----------------------------
df_tft = df_final.copy()  # your DataFrame

# Reset index and create numeric time index
df_tft = df_tft.reset_index().rename(columns={'index': 'date'})
df_tft[TIME_IDX] = np.arange(len(df_tft))

# Add group id for TFT (needed even for single series)
df_tft['series'] = 'EURUSD'

# -----------------------------
# 3. Define TimeSeriesDataSet
# -----------------------------
time_varying_known_reals = [
    'US_Interest_Rate_D1', 'US_Inflation_D1', 'US_Unemployment_D1',
    'US_NonFarm_Payrolls_D1', 'US_GDP_D1', 'US_Trade_Balance_D1',
    'Global_VIX', 'WTI_Crude_Oil_Price','news_sentiment_directional_D1', TIME_IDX
]

time_varying_unknown_reals = [TARGET]

# Split training and validation
training_cutoff = int(len(df_tft) * 0.8)

training = TimeSeriesDataSet(
    df_tft[lambda x: x[TIME_IDX] <= training_cutoff],
    time_idx=TIME_IDX,
    target=TARGET,
    group_ids=['series'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df_tft, predict=True, stop_randomization=True
)

# -----------------------------
# 4. Create Dataloaders
# -----------------------------
train_dataloader = training.to_dataloader(batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = validation.to_dataloader(batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# 5. Define TFT Model
# -----------------------------
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=16,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,  # 7 quantiles
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

# -----------------------------
# 6. Trainer
# -----------------------------
trainer = Trainer(
    max_epochs=N_EPOCHS,
    accelerator="auto",
    gradient_clip_val=0.1,
    enable_progress_bar=True,
)

# -----------------------------
# 7. Fit model
# -----------------------------
trainer.fit(
    model=tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

# -----------------------------
# 8. Forecast
# -----------------------------
y_pred = tft.predict(val_dataloader)  # predictions

# Extract actual target values
y_true = torch.cat([y[0] for x, y in val_dataloader], dim=0).numpy()
y_true_tft = y_true
y_pred_tft = y_pred
rmse = np.sqrt(((y_true - y_pred.numpy())**2).mean())
print(f"\nTFT Forecast RMSE: {rmse:.6f}")
# Median prediction
y_pred_median_tft = tft.predict(val_dataloader)[:, 0]

# True values
y_true_all_tft = torch.cat([y[0][:, 0] for x, y in val_dataloader], dim=0).numpy()



C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](htt

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\loops\fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 5/5 [00:00<00:00,  8.67it/s, v_num=44, train_loss_step=0.0248]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 5/5 [00:00<00:00,  9.03it/s, v_num=44, train_loss_step=0.0231, val_loss=0.011, train_loss_epoch=0.0255]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 5/5 [00:00<00:00, 10.65it/s, v_num=44, train_loss_step=0.017, val_loss=0.00798, train_loss_epoch=0.0199] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 5/5 [00:00<00:00, 10.29it/s, v_num=44, train_loss_step=0.0206, val_loss=0.00756, train_loss_epoch=0.0177]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 5/5 [00:00<00:00,  9.94it/s, v_num=44, train_loss_step=0.0151, val_loss=0.00741, train_loss_epoch=0.0151]
Validation: |    

...

Epoch 29: 100%|██████████| 5/5 [00:01<00:00,  2.85it/s, v_num=44, train_loss_step=0.0149, val_loss=0.00596, train_loss_epoch=0.0137]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.



TFT Forecast RMSE: 0.002585


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


In [44]:
# Save checkpoint
checkpoint_path = "tft_model_sentiment.ckpt"
trainer.save_checkpoint(checkpoint_path)
print("Model saved:", checkpoint_path)


Model saved: tft_model_sentiment.ckpt


## 4\. 📈 Model Evaluation and Validation

The model's performance must be rigorously tested before it can be trusted for decision-making.

  * **Action:** Evaluate the trained model's forecasting accuracy on the unseen **test set** using appropriate metrics, such as **Mean Squared Error (MSE)**, **Root Mean Squared Error (RMSE)**, or directional accuracy (predicting the correct up/down movement).
  * **Explanation:** The final model should demonstrate robust, out-of-sample prediction performance that is significantly better than a simple **random walk** or "no change" benchmark.


In [45]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

def evaluate_forecast(y_true, y_pred, target_name="Target"):
    """
    Evaluates forecasting performance including RMSE, MSE, R², and directional accuracy.
    """
    # Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)
    
    # Root Mean Squared Error
    rmse = np.sqrt(mse)
    
    # R-squared
    r2 = r2_score(y_true, y_pred)
    
    # Directional Accuracy (% of correct up/down movements)
    directional_true = np.sign(y_true)
    directional_pred = np.sign(y_pred)
    directional_accuracy = np.mean(directional_true == directional_pred)
    
    # Naive benchmark (predict no change, i.e., always 0)
    naive_pred = np.zeros_like(y_true)
    naive_mse = mean_squared_error(y_true, naive_pred)
    naive_rmse = np.sqrt(naive_mse)
    naive_r2 = r2_score(y_true, naive_pred)
    
    print(f"\n--- Model Evaluation for {target_name} ---")
    print(f"RMSE: {rmse:.6f}")
    print(f"MSE: {mse:.6f}")
    print(f"R²: {r2:.4f}")
    print(f"Directional Accuracy: {directional_accuracy*100:.2f}%")
    
    print(f"\nNaive Benchmark (No-change) Performance:")
    print(f"RMSE: {naive_rmse:.6f}, R²: {naive_r2:.4f}")
    
    return {
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "directional_accuracy": directional_accuracy,
        "naive_mse": naive_mse,
        "naive_rmse": naive_rmse,
        "naive_r2": naive_r2
    }

# --- Example Usage with Random Forest Predictions ---
# y_test: true values
# rf_pred: predicted values from RF

eval_results = evaluate_forecast(y_test_rf, rf_pred, target_name="EURUSD=X_D1")



--- Model Evaluation for EURUSD=X_D1 ---
RMSE: 0.021482
MSE: 0.000461
R²: -0.0017
Directional Accuracy: 53.06%

Naive Benchmark (No-change) Performance:
RMSE: 0.021465, R²: -0.0001


In [46]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import torch

def evaluate_tft_forecast(y_true, y_pred, target_name="Target", quantile_index=0):
    """
    Evaluate TFT forecast.
    
    Parameters:
    - y_true: np.array or torch.Tensor of true values
    - y_pred: np.array, torch.Tensor, or TFT prediction array (N x Q quantiles)
    - target_name: string, name of the target
    - quantile_index: int, which quantile to use as point estimate (default 0=median if 7 quantiles)
    
    Returns: dict of evaluation metrics
    """
    # Ensure numpy arrays
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.numpy()
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.numpy()
    
    # If prediction has multiple quantiles, pick one (default median)
    if y_pred.ndim > 1:
        y_pred_point = y_pred[:, quantile_index]
    else:
        y_pred_point = y_pred
    
    # Metrics
    mse = mean_squared_error(y_true, y_pred_point)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred_point)
    
    # Directional accuracy
    directional_true = np.sign(np.diff(y_true, prepend=y_true[0]))
    directional_pred = np.sign(np.diff(y_pred_point, prepend=y_pred_point[0]))
    directional_accuracy = np.mean(directional_true == directional_pred)
    
    # Naive benchmark (no-change prediction)
    naive_pred = y_true[:-1].tolist() + [y_true[-1]]  # last value stays
    naive_mse = mean_squared_error(y_true, naive_pred)
    naive_rmse = np.sqrt(naive_mse)
    naive_r2 = r2_score(y_true, naive_pred)
    
    print(f"\n--- TFT Model Evaluation for {target_name} ---")
    print(f"RMSE: {rmse:.6f}")
    print(f"MSE: {mse:.6f}")
    print(f"Directional Accuracy: {directional_accuracy*100:.2f}%")
    
    print(f"\nNaive Benchmark (No-change) Performance:")
    print(f"RMSE: {naive_rmse:.6f}")
    
    return {
        "mse": mse,
        "rmse": rmse,

        "directional_accuracy": directional_accuracy,
        "naive_mse": naive_mse,
        "naive_rmse": naive_rmse,
    
    }

# --- Usage ---
eval_results_tft = evaluate_tft_forecast(y_true_all_tft, y_pred_median_tft, target_name="EURUSD=X_D1")



--- TFT Model Evaluation for EURUSD=X_D1 ---
RMSE: 0.002585
MSE: 0.000007
Directional Accuracy: 100.00%

Naive Benchmark (No-change) Performance:
RMSE: 0.000000


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


### INFERENCE

In [11]:
# import pandas as pd
# import pandas_datareader.data as web
# import yfinance as yf
# import pandas_ta_classic as ta
# import numpy as np
# import torch
# from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
# from datetime import datetime
# from dateutil.relativedelta import relativedelta

# # --- 1. CONFIGURATION ---
# MODEL_PATH = "tft_model.ckpt"
# CURRENCY_PAIR = 'EURUSD=X'
# FRED_CODES = {
#     'FEDFUNDS': 'US_Interest_Rate',
#     'CPIAUCSL': 'US_Inflation',
#     'UNRATE': 'US_Unemployment',
#     'PAYEMS': 'US_NonFarm_Payrolls',
#     'GDPC1': 'US_GDP',
#     'BOPGSTB': 'US_Trade_Balance',
#     'VIXCLS': 'Global_VIX',
#     'DCOILWTICO': 'WTI_Crude_Oil_Price'
# }
# TARGET_COL = 'EURUSD=X' # Note: In the logic below, we rename this to Close_Price
# FREQ = 'M'

# class ForexForecaster:
#     def __init__(self, model_path):
#         print(f"Loading model from {model_path}...")
#         # Load on CPU
#         self.model = TemporalFusionTransformer.load_from_checkpoint(model_path, map_location=torch.device('cpu'))
#         self.model.eval()
        
#     def fetch_recent_data(self, lookback_months=60):
#         """
#         Fetches historical data and ensures strict column naming.
#         """
#         end_date = datetime.today()
#         start_date = end_date - relativedelta(months=lookback_months)
        
#         print(f"Fetching live data from {start_date.date()} to {end_date.date()}...")

#         # 1. Fetch Forex (Robust Method)
#         # auto_adjust=True removes the warning and handles splits/dividends (though rare for Forex)
#         raw_yfinance = yf.download(CURRENCY_PAIR, start=start_date, end=end_date, interval='1d', progress=False, auto_adjust=True)
        
#         # Handle yfinance MultiIndex columns (The root cause of your error)
#         # Check if columns have levels (Price Type, Ticker)
#         if isinstance(raw_yfinance.columns, pd.MultiIndex):
#             try:
#                 # Try to extract 'Close'
#                 df_fx = raw_yfinance.xs('Close', level=0, axis=1)
#             except KeyError:
#                 # Fallback if 'Close' isn't top level
#                 df_fx = raw_yfinance['Close']
#         else:
#             df_fx = raw_yfinance[['Close']]

#         # Force valid DataFrame structure
#         if isinstance(df_fx, pd.Series):
#             df_fx = df_fx.to_frame()
            
#         # If the column is still named strictly after the ticker, rename it
#         # We assume the first column is the Close price
#         df_fx = df_fx.iloc[:, [0]] 
#         df_fx.columns = ["Close_Price"]
        
#         # Remove Timezone info to match FRED data (FRED is usually naive)
#         df_fx.index = df_fx.index.tz_localize(None)

#         # 2. Fetch Macro (FRED)
#         try:
#             df_fred = web.DataReader(list(FRED_CODES.keys()), 'fred', start_date, end_date)
#             df_fred.columns = list(FRED_CODES.values())
#         except Exception as e:
#             print(f"Warning: FRED fetch failed ({e}). Using zeros.")
#             df_fred = pd.DataFrame(columns=FRED_CODES.values())

#         # 3. Combine
#         df = df_fx.join(df_fred, how='outer')
        
#         # 4. Fill NaNs
#         df = df.ffill().bfill()
        
#         # Debug print to ensure column exists
#         if 'Close_Price' not in df.columns:
#             print("CRITICAL ERROR: 'Close_Price' column missing after join.")
#             print(f"Columns found: {df.columns}")
        
#         return df

#     def preprocess_for_inference(self, df):
#         df = df.copy()

#         # Check for column existence before proceeding
#         if 'Close_Price' not in df.columns:
#             raise KeyError(f"'Close_Price' not found. Available columns: {df.columns.tolist()}")

#         # --- A. Technical Indicators ---
#         df['Lag_1D'] = df['Close_Price'].shift(1)
#         df['SMA_20'] = ta.sma(df['Close_Price'], length=20)
#         df['SMA_50'] = ta.sma(df['Close_Price'], length=50)
#         df['RSI_14'] = ta.rsi(df['Close_Price'], length=14)
#         df['Historical_Volatility_20D'] = df['Close_Price'].pct_change().rolling(window=20).std() * (252**0.5)

#         # --- B. Resample to Monthly ---
#         # Initialize resampled dataframe
#         df_res = pd.DataFrame()
        
#         # Resample Close_Price specifically
#         df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
        
#         # Resample other columns
#         # Note: We skip 'Close_Price' in the loop because we renamed it to 'EURUSD=X' above 
#         # to match the Target name expected by the model during training.
#         skip_cols = ['Close_Price']
        
#         for col in df.columns:
#             if col not in skip_cols:
#                 df_res[col] = df[col].resample(FREQ).last()
        
#         df_res = df_res.ffill().dropna()

#         # --- C. Feature Engineering (Lags) ---
#         numeric_cols = df_res.select_dtypes(include=np.number).columns
#         for col in numeric_cols:
#             df_res[f'{col}_Lag1'] = df_res[col].shift(1)
#             df_res[f'{col}_Lag4'] = df_res[col].shift(4)

#         # --- D. Stationarity (Differencing) ---
#         # Capture last state for reconstruction
#         last_actual_price = df_res['EURUSD=X'].iloc[-1]
#         last_actual_date = df_res.index[-1]

#         # Apply differencing to non-lagged columns
#         cols_to_diff = [c for c in df_res.columns if 'Lag' not in c] 
        
#         for col in cols_to_diff:
#             df_res[f'{col}_D1'] = df_res[col].diff()

#         # Drop NaNs created by lags/diffs
#         df_res = df_res.dropna()

#         return df_res, last_actual_price, last_actual_date

#     def predict_next_month(self):
#         # 1. Get Data
#         raw_df = self.fetch_recent_data()
        
#         # 2. Process
#         processed_df, last_price, last_date = self.preprocess_for_inference(raw_df)
        
#         print(f"\nContext: Last closed month data available: {last_date.date()}")
#         print(f"Context: Last Price: {last_price:.4f}")

#         # 3. Prepare for PyTorch Forecasting
#         # Create Dummy Row for next month to provide a target container
#         new_row = processed_df.iloc[[-1]].copy()
#         new_row.index = [last_date + relativedelta(months=1)]
        
#         inference_df = pd.concat([processed_df, new_row])
        
#         # Add required TFT columns
#         inference_df['time_idx'] = np.arange(len(inference_df))
#         inference_df['series'] = 'EURUSD'
#         inference_df = inference_df.reset_index().rename(columns={'index': 'date'})

#         # 4. Create Prediction Dataset
#         try:
#             # --- FIX IS HERE ---
#             # Use 'from_parameters' instead of 'from_dataset' because dataset_parameters is a dict
#             pred_ds = TimeSeriesDataSet.from_parameters(
#                 self.model.dataset_parameters, 
#                 inference_df, 
#                 predict=True, 
#                 stop_randomization=True
#             )
            
#             # Create dataloader
#             pred_loader = pred_ds.to_dataloader(batch_size=1, shuffle=False)
            
#             # 5. Predict
#             # return_x=False returns only the predictions
#             raw_prediction = self.model.predict(pred_loader, mode="prediction", return_x=False)
            
#             # Extract the actual value (Batch 0, Step 0)
#             predicted_change = raw_prediction[0][0].item() 
            
#             # 6. Reconstruct Price
#             predicted_price = last_price + predicted_change
            
#             return {
#                 "target_date": (last_date + relativedelta(months=1)).date(),
#                 "last_price": last_price,
#                 "predicted_change": predicted_change,
#                 "predicted_price": predicted_price
#             }

#         except Exception as e:
#             print(f"Inference Error: {e}")
#             import traceback
#             traceback.print_exc()
#             return None
# # --- EXECUTION ---
# if __name__ == "__main__":
#     print("--- starting inference pipeline ---")
#     forecaster = ForexForecaster(MODEL_PATH)
#     result = forecaster.predict_next_month()
    
#     if result:
#         print("\n" + "="*40)
#         print(f"FORECAST REPORT FOR: {result['target_date'].strftime('%B %Y')}")
#         print("="*40)
#         print(f"Current Price:          {result['last_price']:.4f}")
#         print(f"Predicted Change:       {result['predicted_change']:.4f}")
#         print("-" * 40)
#         print(f"FINAL PREDICTION:       {result['predicted_price']:.4f}")
#         print("="*40)

--- starting inference pipeline ---
Loading model from tft_model.ckpt...


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Fetching live data from 2020-11-27 to 2025-11-27...


C:\Users\Vipin\AppData\Local\Temp\ipykernel_49464\3689960390.py:111: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_49464\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_49464\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_49464\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_49464\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a 


Context: Last closed month data available: 2025-11-30
Context: Last Price: 1.1602

FORECAST REPORT FOR: December 2025
Current Price:          1.1602
Predicted Change:       -0.0032
----------------------------------------
FINAL PREDICTION:       1.1570


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.


## 5\. 🖥️ Interactive Dashboard Creation

Once the model is finalized, the predictions are packaged for end-user interpretation.

  * **Action:** Develop a front-end interface (dashboard) using a visualization tool (e.g., Power BI, Tableau) or a web framework (e.g., Python's Streamlit, Dash, or a custom stack).
  * **Explanation:** The dashboard should display the model's key outputs:
      * The **forecasted currency movement** (e.g., a trend line).
      * The **confidence interval** or prediction uncertainty.
      * A breakdown of **feature importance** (which macroeconomic/geopolitical indicators had the largest impact on the prediction).
      * User-friendly controls to select currencies or time horizons.


## 6\. 🚀 Deployment and Monitoring(we are not doing this step)

The final stage is integrating the model and dashboard into a live, usable system.

  * **Action:** Deploy the model's prediction pipeline to a production environment where it can be regularly updated with new data and generate real-time forecasts. Set up monitoring to track the model's performance over time (**model drift**) and ensure the data pipeline remains stable.
  * **Explanation:** A model's performance can degrade as market dynamics change. Continuous monitoring and retraining are necessary to maintain predictive accuracy and provide reliable information to the end-users accessing the interactive dashboard.
